# 🕯️ Candlestick ViT — Model Training Notebook

This notebook trains the Vision Transformer (ViT) model for candlestick chart trend prediction.

**Requirements:**
- Google Colab with GPU (T4 or A100)
- Training data (candlestick chart images + labels CSV)
- Google Drive mounted for checkpoint persistence

---

## 1. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Create project directory
import os
PROJECT_DIR = '/content/drive/MyDrive/candlestick_vit'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f'{PROJECT_DIR}/checkpoints', exist_ok=True)
print(f'Project directory: {PROJECT_DIR}')

In [ ]:
# Install dependencies
!pip install -q torch torchvision timm scikit-learn pandas tqdm pillow matplotlib seaborn

In [ ]:
# Check GPU availability
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Upload Training Data

Upload your training data to Google Drive at:
- `candlestick_vit/data/charts/` — Chart images organized by ticker
- `candlestick_vit/data/labeled_dataset.csv` — Labels CSV

Or upload directly from your local machine:

In [ ]:
# Option 1: Upload from local machine
# from google.colab import files
# uploaded = files.upload()  # Upload labeled_dataset.csv

# Option 2: Use data already in Google Drive
DATA_DIR = f'{PROJECT_DIR}/data'
LABELS_CSV = f'{DATA_DIR}/labeled_dataset.csv'

# Check if data exists
if os.path.exists(LABELS_CSV):
    import pandas as pd
    df = pd.read_csv(LABELS_CSV)
    print(f'Dataset loaded: {len(df)} samples')
    print(f'Splits: {df["split"].value_counts().to_dict()}')
    print(f'Labels: {df["label"].value_counts().to_dict()}')
else:
    print(f'⚠️ Data not found at {LABELS_CSV}')
    print('Please upload your labeled_dataset.csv and chart images to Google Drive.')

## 3. Dataset & Model Definition

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
from PIL import Image
import pandas as pd
import numpy as np
import json
import time
from tqdm import tqdm

# Configuration
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.01
EARLY_STOPPING_PATIENCE = 5
SCHEDULER_T_MAX = 10
NUM_CLASSES = 3
CLASS_NAMES = ['Down', 'Neutral', 'Up']
VIT_MODEL_NAME = 'vit_base_patch16_224'
CHECKPOINT_PATH = f'{PROJECT_DIR}/checkpoints/vit_candlestick_best.pth'

print('Configuration loaded.')

In [ ]:
class CandlestickDataset(Dataset):
    """PyTorch Dataset for candlestick chart images."""
    
    def __init__(self, csv_path, split='train', transform=None):
        df = pd.read_csv(csv_path)
        self.data = df[df['split'] == split].reset_index(drop=True)
        self.transform = transform
        print(f'Loaded {split} set: {len(self.data)} samples')
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['label_id'])
        if self.transform:
            image = self.transform(image)
        return image, label


class CandlestickViT(nn.Module):
    """Vision Transformer for candlestick chart classification."""
    
    def __init__(self, model_name=VIT_MODEL_NAME, num_classes=NUM_CLASSES, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        self.feature_dim = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.LayerNorm(self.feature_dim),
            nn.Dropout(0.1),
            nn.Linear(self.feature_dim, 256),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        features = self.backbone(x)
        return self.classifier(features)


print('Classes defined.')

## 4. Data Transforms & Loaders

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomRotation(degrees=5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Create datasets
train_dataset = CandlestickDataset(LABELS_CSV, 'train', train_transform)
val_dataset = CandlestickDataset(LABELS_CSV, 'val', val_transform)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

## 5. Training

In [ ]:
# Setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

model = CandlestickViT(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=SCHEDULER_T_MAX)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')

In [ ]:
# Training loop
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
best_val_loss = float('inf')
patience_counter = 0
best_epoch = 0

print(f'Starting training for {NUM_EPOCHS} epochs...')
print('=' * 80)

start_time = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    # Train
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    
    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Train]', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        train_correct += predicted.eq(labels).sum().item()
        train_total += labels.size(0)
    
    train_loss /= train_total
    train_acc = train_correct / train_total
    
    # Validate
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch}/{NUM_EPOCHS} [Val]', leave=False):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)
    
    val_loss /= val_total
    val_acc = val_correct / val_total
    
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()
    
    # Record
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)
    
    print(f'Epoch [{epoch:3d}/{NUM_EPOCHS}] '
          f'| Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} '
          f'| Val Loss: {val_loss:.4f} Acc: {val_acc:.4f} '
          f'| LR: {current_lr:.6f}')
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        patience_counter = 0
        
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_acc': val_acc,
            'train_loss': train_loss,
            'train_acc': train_acc,
        }
        torch.save(checkpoint, CHECKPOINT_PATH)
        print(f'  💾 Best model saved (val_loss: {val_loss:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f'\n⏹️ Early stopping at epoch {epoch}')
            break

total_time = time.time() - start_time
print(f'\n{"="*80}')
print(f'Training complete in {total_time:.1f}s ({total_time/60:.1f}min)')
print(f'Best epoch: {best_epoch}, Best val loss: {best_val_loss:.4f}')
print(f'Checkpoint saved to: {CHECKPOINT_PATH}')

## 6. Training Visualization

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0].plot(epochs_range, history['train_loss'], 'b-', label='Train')
axes[0].plot(epochs_range, history['val_loss'], 'r-', label='Validation')
axes[0].set_title('Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(epochs_range, history['train_acc'], 'g-', label='Train')
axes[1].plot(epochs_range, history['val_acc'], 'orange', label='Validation')
axes[1].set_title('Accuracy', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning Rate
axes[2].plot(epochs_range, history['lr'], 'purple')
axes[2].set_title('Learning Rate', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/training_curves.png', dpi=150)
plt.show()

# Save history
with open(f'{PROJECT_DIR}/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Training history saved.')

## 7. Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# Load best checkpoint
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Test set
test_dataset = CandlestickDataset(LABELS_CSV, 'test', val_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc='Testing'):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print('\n' + classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/confusion_matrix.png', dpi=150)
plt.show()

## 8. Download Checkpoint

Download the trained model checkpoint to use locally.

In [ ]:
from google.colab import files

print(f'Checkpoint location: {CHECKPOINT_PATH}')
print(f'File size: {os.path.getsize(CHECKPOINT_PATH) / 1e6:.1f} MB')

# Download to local machine
files.download(CHECKPOINT_PATH)
print('\n✅ Download started! Place this file in: models/checkpoints/vit_candlestick_best.pth')